### Cell 1 — Environment & Project Validation

This cell automatically detects the correct Maven project root by locating the pom.xml file, then identifies all src/main/java and src/test/java directories (including multi-module setups). It also counts the Java source files found and verifies that Maven is installed and accessible in the environment.

In [1]:
# ============================================================
# STEP 1 v3 — AUTO-DETECT MAVEN ROOT + ALL JAVA SOURCE ROOTS
#   - Finds the correct PROJECT_ROOT (folder containing pom.xml)
#   - Finds ALL */src/main/java and */src/test/java under it
#   - Counts Java files across all detected source roots
# ============================================================

import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ---- Base workspace path (adjust only this if needed) ----
# Use the workspace root or a parent directory containing the Maven project.
# The old hard‑coded path pointed at a different repository; update it to
# match our current workspace. Using Path.cwd() would work if the notebook
# is launched from the repo root.
BASE_PATH = Path("/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb").resolve()
# Alternatively: BASE_PATH = Path.cwd()

print(f"Scanning inside: {BASE_PATH}")

# ---- Auto-detect Maven project root (pom.xml) ----
pom_files = list(BASE_PATH.rglob("pom.xml"))
if not pom_files:
    raise Exception("❌ No pom.xml found inside BASE_PATH")

# For projects with multiple modules you could filter by directory name
# or other heuristics. Here there is only one pom under facade/ so just
# use the first match.
PROJECT_ROOT = pom_files[0].parent

print(f"\n✅ Maven project detected at:\n{PROJECT_ROOT}")

# ---- Detect ALL Java source roots (supports multi-module) ----
main_java_roots = sorted({p.parent for p in PROJECT_ROOT.rglob("src/main/java")})
test_java_roots = sorted({p.parent for p in PROJECT_ROOT.rglob("src/test/java")})

print("\n[Detected source roots]")
print("Main Java roots:")
for r in main_java_roots:
    print(" -", r.relative_to(PROJECT_ROOT))

print("\nTest Java roots:")
for r in test_java_roots:
    print(" -", r.relative_to(PROJECT_ROOT))

# ---- Count Java files across all main roots ----
main_java_files = []
for root in main_java_roots:
    main_java_files.extend(list(root.rglob("*.java")))

test_java_files = []
for root in test_java_roots:
    test_java_files.extend(list(root.rglob("*.java")))

print(f"\nTotal MAIN Java files detected: {len(main_java_files)}")
print(f"Total TEST Java files detected: {len(test_java_files)}")

print("\nSample MAIN Java files:")
for f in main_java_files[:8]:
    print("-", f.relative_to(PROJECT_ROOT))

# ---- Check Maven availability ----
print("\n[Checking Maven installation]")
result = subprocess.run(["mvn", "-v"], capture_output=True, text=True)
print(result.stdout.split("\n")[0])

Scanning inside: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb

✅ Maven project detected at:
/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade

[Detected source roots]
Main Java roots:
 - .ai_artifacts/20260303_232044/generated_files/src/main
 - .ai_artifacts/20260303_232217/generated_files/src/main
 - .ai_artifacts/20260303_234548/generated_files/src/main
 - .ai_artifacts/20260303_234623/generated_files/src/main
 - .ai_artifacts/20260303_234835/generated_files/src/main
 - .ai_artifacts/20260303_235035/generated_files/src/main
 - .ai_artifacts/20260304_000054/generated_files/src/main
 - .ai_artifacts/20260304_000135/generated_files/src/main
 - .ai_artifacts/20260304_000315/generated_files/src/main
 - .ai_artifacts/20260304_000604/generated_files/src/main
 - .ai_artifacts/20260304_085323/generated_files/src/main
 - .ai_artifacts/20260304_085352/generated_files/src/main
 - src/main

Test Java roots:
 - .ai_artifacts/20260303_232044/generated_files/src/test
 - .ai_artifacts/2

###  Cell 2 -  Baseline Build (Ground Truth)

This cell runs a baseline mvn clean test build to establish the project’s ground truth before any AI modifications. It captures the exit code and recent logs, ensuring the project compiles and tests pass so you know any later failures are caused by AI-generated changes, not pre-existing issues.

In [2]:
# ==========================================
# STEP 2 — BASELINE BUILD (GROUND TRUTH)
# ==========================================

import subprocess
from datetime import datetime

print(f"Running baseline build in: {PROJECT_ROOT}")
print("Command: mvn clean test\n")

start = datetime.now()

result = subprocess.run(
    ["mvn", "clean", "test"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

end = datetime.now()
duration = (end - start).total_seconds()

print(f"Exit code: {result.returncode}")
print(f"Duration: {duration:.2f}s")

# Show last part of logs (most useful)
print("\n--- STDOUT (last 80 lines) ---")
stdout_lines = result.stdout.splitlines()
print("\n".join(stdout_lines[-80:]))

print("\n--- STDERR (last 80 lines) ---")
stderr_lines = result.stderr.splitlines()
print("\n".join(stderr_lines[-80:]))

# Simple status
if result.returncode == 0:
    print("\n✅ Baseline build PASSED. Safe to proceed to AI steps.")
else:
    print("\n❌ Baseline build FAILED. Fix project/env first before adding AI.")

Running baseline build in: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade
Command: mvn clean test

Exit code: 0
Duration: 11.33s

--- STDOUT (last 80 lines) ---
      Did not match:
         - @ConditionalOnClass did not find required class 'org.springframework.security.config.http.SessionCreationPolicy' (OnClassCondition)

   SimpleCacheConfiguration:
      Did not match:
         - Cache org.springframework.boot.autoconfigure.cache.SimpleCacheConfiguration unknown cache type (CacheCondition)

   TaskExecutorConfigurations.SimpleAsyncTaskExecutorBuilderConfiguration#simpleAsyncTaskExecutorBuilderVirtualThreads:
      Did not match:
         - @ConditionalOnMissingBean (types: org.springframework.boot.task.SimpleAsyncTaskExecutorBuilder; SearchStrategy: all) found beans of type 'org.springframework.boot.task.SimpleAsyncTaskExecutorBuilder' simpleAsyncTaskExecutorBuilder (OnBeanCondition)

   TaskExecutorConfigurations.TaskExecutorConfiguration#applicationTaskExecutorVirtualT

### Cell 3 - MINIMAL CONTEXT LOADER (NO RAG)

This cell builds a lightweight code context: it indexes all Java files, then selects a small relevant subset using keyword matching (seed files) and expands it by scanning those files for referenced class names to pull in likely dependencies. The goal is to feed the LLM only the most relevant files (plus pom.xml) instead of the whole repo, to save tokens and reduce noise.

its a 2-step filtering process: 
    - by key words
    - imported based / CamelCase

In [3]:
# ==========================================
# STEP 3 (v2) — MINIMAL CONTEXT + SMART PICKING
#   - Index java files
#   - Pick by keywords
#   - Expand context by scanning imports/usages (cheap heuristic)
# ==========================================

import re
from pathlib import Path

def read_text_file(path: Path, max_chars: int = 120_000) -> str:
    if not path.exists():
        return ""
    txt = path.read_text(encoding="utf-8", errors="replace")
    return txt[:max_chars]

# Base context
context = {}
context["pom.xml"] = read_text_file(PROJECT_ROOT / "pom.xml", max_chars=120_000) # Add pom.xml to context for better dependency understanding

# Index all main Java files (paths only)
java_index = []
for root in main_java_roots:
    java_index.extend(list(root.rglob("*.java")))
java_index = sorted(java_index)

# Build a map: ClassName -> path (simple: filename without .java)
class_to_path = {p.stem: p for p in java_index}

print(f"Indexed Java files (main): {len(java_index)}")

def pick_files_by_keywords(keywords, limit=8):
    kws = [k.lower() for k in keywords]
    matches = []
    for f in java_index:
        p = str(f).lower()
        if any(k in p for k in kws):
            matches.append(f)
    return matches[:limit]

def expand_by_references(seed_files, max_extra=6):
    """
    Cheap dependency expansion:
    - read seed files
    - find words that look like ClassNames (CamelCase)
    - if a ClassName exists in project, include its file too
    """
    extra = []
    seen = {sf for sf in seed_files}
    for sf in seed_files:
        txt = read_text_file(sf, max_chars=30_000)
        # find CamelCase tokens (very rough)
        candidates = set(re.findall(r"\b[A-Z][A-Za-z0-9_]{2,}\b", txt))
        for c in candidates:
            if c in class_to_path:
                p = class_to_path[c]
                if p not in seen:
                    extra.append(p)
                    seen.add(p)
                    if len(extra) >= max_extra:
                        return seed_files + extra
    return seed_files + extra

def expand_by_imports(seed_files, max_extra=6):
    """
    More precise dependency expansion:
    - Reads import statements from seed files
    - If imported class exists inside project, include its file
    """
    extra = []
    seen = set(seed_files)

    for sf in seed_files:
        txt = read_text_file(sf, max_chars=40_000)

        # Extract import lines
        imports = re.findall(r"import\s+([\w\.]+);", txt)

        for imp in imports:
            class_name = imp.split(".")[-1]  # get final class name
            if class_name in class_to_path:
                p = class_to_path[class_name]
                if p not in seen:
                    extra.append(p)
                    seen.add(p)
                    if len(extra) >= max_extra:
                        return seed_files + extra

    return seed_files + extra

# Example (adjust keywords to your project domain before running)
# This repository deals with adherents, events, recettes, etc.; pick some
# representative terms so the context loader finds relevant controllers.
seed = pick_files_by_keywords(["adherent", "event", "recette"], limit=5)
# picked_paths = expand_by_references(seed, max_extra=6)
picked_paths = expand_by_imports(seed, max_extra=6)


print("\nPicked files (seed + expanded):")
for f in picked_paths:
    print("-", f.relative_to(PROJECT_ROOT))

Indexed Java files (main): 40

Picked files (seed + expanded):
- src/main/java/n7/facade/Adherent.java
- src/main/java/n7/facade/AdherentController.java
- src/main/java/n7/facade/AdherentRepository.java
- src/main/java/n7/facade/Event.java
- src/main/java/n7/facade/EventRepository.java


In [4]:
import os, sys, requests, json
from dotenv import load_dotenv
load_dotenv(override=True)

k = os.getenv("OPENROUTER_API_KEY")
b = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
m = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

print("python:", sys.executable)
print("key head/tail:", k[:12], k[-8:], "len=", len(k))
print("base:", b, "model:", m)

r = requests.post(
    f"{b}/chat/completions",
    headers={
        "Authorization": f"Bearer {k}",
        "Content-Type": "application/json",
    },
    json={
        "model": m,
        "messages": [{"role": "user", "content": "Reply exactly OK"}],
    },
    timeout=30,
)

print("status:", r.status_code)
print("body:", r.text[:500])


python: /usr/bin/python3
key head/tail: sk-or-v1-f2b 22aa4cfc len= 73
base: https://openrouter.ai/api/v1 model: openai/gpt-4o-mini
status: 200
body: {"id":"gen-1772632136-ZjVbq49ffL9vIr18x5n3","object":"chat.completion","created":1772632136,"model":"openai/gpt-4o-mini","provider":"OpenAI","system_fingerprint":"fp_373a14eb6f","choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"OK","refusal":null,"reasoning":null}}],"usage":{"prompt_tokens":10,"completion_tokens":1,"total_tokens":11,"cost":0.0000021,"is_byok":false,"prompt_tokens_details":{"cached_tokens":0,"cache_


### Cell 4 — Plan-only LLM (no code writing)

This cell asks the LLM to analyze the selected project context and produce a structured implementation plan (JSON only) for a requested feature — without generating any code yet.

It forces the model to:

- Respect existing APIs (no hallucinated methods),

- Specify which files to create/modify,

- Define tests, API contract, risks, and acceptance checks  (before moving to code generation)

In [5]:
# ==========================================
# STEP 4 (v2) — PLAN-ONLY LLM (STRICT, NO HALLUCINATED APIS)
# ==========================================

import os
import json
# Robust imports: prefer provider-specific packages but fall back to main langchain
try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
except Exception:
    try:
        # Newer langchain: chat_models provides ChatOpenAI wrapper
        from langchain.chat_models import ChatOpenAI
    except Exception:
        ChatOpenAI = None
    try:
        # Prompt templates live under langchain.prompts in many installs
        from langchain.prompts import ChatPromptTemplate
    except Exception:
        ChatPromptTemplate = None
    if ChatOpenAI is None or ChatPromptTemplate is None:
        print("⚠️ Warning: LangChain import fallbacks unavailable. Ensure 'langchain' or 'langchain_openai' is installed.")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
# MODEL_NAME = os.getenv("MODEL_NAME", "anthropic/claude-3.5-sonnet")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY in your .env")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.2,
    max_tokens=650,   # keep low for budget
)

# written by dev
#FEATURE_REQUEST = """
#Add a REST endpoint to fetch a student's progress summary.
#Return: studentId, totalPoints, level, badgesCount.
#Include basic validation and proper HTTP status codes.
#""" 
 
# written by dev
FEATURE_REQUEST = """
Add a minimal proof-of-concept REST endpoint to validate the AI pipeline.

API:
- Method: GET
- Path: /ai/poc
- Success response (200) JSON:
  {
    "status": "ok",
    "timestamp": "<ISO-8601 string>"
  }

Validation:
- If a query param `bad=true` is provided, return HTTP 400 with JSON:
  { "error": "bad request" }

HARD CONSTRAINTS (must be enforced):
- Do NOT access the database.
- Do NOT create or modify any Service/Repository/DAO classes.
- Do NOT modify existing project files (files_to_modify MUST be []).
- Only allowed changes: create EXACTLY these two new files:
  1) src/main/java/n7/facade/AiPocController.java
  2) src/test/java/n7/facade/AiPocControllerWebMvcTest.java

Testing:
- Add a WebMvcTest (MockMvc) that checks:
  - GET /ai/poc returns 200 and JSON has keys status,timestamp (status == "ok")
  - GET /ai/poc?bad=true returns 400 and JSON contains key error

Goal:
- Minimal change. No integration with existing features. Just prove the pipeline works end-to-end.
"""

# Load only picked files (from Cell 3 v2)
picked_files = {}
for p in picked_paths[:10]:
    picked_files[str(p.relative_to(PROJECT_ROOT))] = read_text_file(p, max_chars=6000)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Spring Boot engineer. Produce ONLY valid JSON. No markdown."),
    ("human",
     """You must plan changes for a Java/Spring Boot Maven project.

CRITICAL RULE:
- You may NOT invent calls to methods/classes that are not present in the provided context.
- If needed methods do not exist, you MUST plan to add them (and plan tests).

Context:
- pom.xml (partial)
{pom}

Picked project code context:
{picked_context}

FEATURE REQUEST:
{feature_request}

Return STRICT JSON with schema:
{{
  "summary": "...",
  "files_to_modify": ["..."],
  "files_to_create": ["..."],
  "tests_to_add": [
    {{"type":"unit|webmvc|integration", "target":"...", "notes":"..."}}
  ],
  "api_contract": {{
    "method":"GET",
    "path":"/...",
    "responses":[{{"status":200,"body_example":{{}}}},{{"status":400,"body_example":{{}}}}]
  }},
  "acceptance_checks": ["mvn test passes", "..."],
  "risks": ["..."]
}}"""
    )
])

msg = prompt.format_messages(
    pom=context["pom.xml"][:1200],
    picked_context=json.dumps({k: v[:2500] for k, v in picked_files.items()}, ensure_ascii=False, indent=2),
    feature_request=FEATURE_REQUEST.strip()
)

resp = llm.invoke(msg).content

try:
    plan = json.loads(resp)
    print("✅ Plan JSON parsed.\n")
    print(json.dumps(plan, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("❌ Invalid JSON. Raw output:\n")
    print(resp)


✅ Plan JSON parsed.

{
  "summary": "Add a proof-of-concept REST endpoint to validate the AI pipeline.",
  "files_to_modify": [],
  "files_to_create": [
    "src/main/java/n7/facade/AiPocController.java",
    "src/test/java/n7/facade/AiPocControllerWebMvcTest.java"
  ],
  "tests_to_add": [
    {
      "type": "webmvc",
      "target": "src/test/java/n7/facade/AiPocControllerWebMvcTest.java",
      "notes": "Test GET /ai/poc for 200 response and valid JSON structure; Test GET /ai/poc?bad=true for 400 response and error JSON."
    }
  ],
  "api_contract": {
    "method": "GET",
    "path": "/ai/poc",
    "responses": [
      {
        "status": 200,
        "body_example": {
          "status": "ok",
          "timestamp": "<ISO-8601 string>"
        }
      },
      {
        "status": 400,
        "body_example": {
          "error": "bad request"
        }
      }
    ]
  },
  "acceptance_checks": [
    "mvn test passes"
  ],
  "risks": [
    "No database interaction, only a simple en

### Cell 5 — Generate Code + Tests (Dry Run, No Saving)

On demande au LLM de décider quels fichiers créer ou modifier (patchset).

Puis on appelle le LLM une fois par fichier pour générer son contenu complet.

On applique des règles de sécurité (pas de markdown, accolades équilibrées, pas de texte parasite).

On stocke les fichiers générés en mémoire (generated_files) sans encore les écrire sur disque.

In [ ]:
# ==========================================
# STEP 5 — EXECUTE PLAN (STRICT PIPELINE)
#   - Uses ONLY `plan` (Cell 4)
#   - Uses ONLY `picked_files` (Cell 3)
#   - Generates tests first
#   - One LLM call per file
#   - Strict safety rules
#   - Stores in memory (generated_files)
# ==========================================

import os, json, re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---- Preconditions ----
if "PROJECT_ROOT" not in globals():
    raise Exception("❌ Run Cell 1 first.")
if "picked_files" not in globals() or not picked_files:
    raise Exception("❌ Run Cell 3 first (picked_files missing).")
if "plan" not in globals() or not isinstance(plan, dict):
    raise Exception("❌ Run Cell 4 first (plan missing).")

if "generated_files" not in globals():
    generated_files = {}

# ---- Model ----
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY.")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.1,
    max_tokens=900,
)

# ---- Safety helpers ----
def strip_fences(s: str) -> str:
    s = s.lstrip()
    s = re.sub(r"```[a-zA-Z0-9_-]*\s*\n", "", s)
    s = s.replace("\n```", "\n")
    return s.strip() + "\n"

def java_safety(rel_path: str, content: str):
    head = content.lstrip()[:200].lower()

    if head.startswith(("file:", "here", "sure", "i'll", "i will")):
        raise Exception(f"❌ {rel_path}: wrapper text detected.")

    if "`" in content[:400] or "```" in content[:400]:
        raise Exception(f"❌ {rel_path}: markdown detected.")

    if rel_path.endswith(".java"):
        if content.count("{") != content.count("}"):
            raise Exception(f"❌ {rel_path}: unbalanced braces (truncated output).")

        h = content.lstrip()[:200]
        if not (
            h.startswith("package ")
            or h.startswith("import ")
            or h.startswith("@")
            or h.startswith("public ")
            or h.startswith("/*")
        ):
            raise Exception(f"❌ {rel_path}: suspicious Java header.")

# ---- Context from Cell 3 ONLY ----
repo_ctx = {
    k: (v or "")[:2000]   # small trim for token safety
    for k, v in list(picked_files.items())[:10]
}

# ---- Targets from plan ----
files_to_create = plan.get("files_to_create", []) or []
files_to_modify = plan.get("files_to_modify", []) or []

targets = []
for p in files_to_create:
    targets.append(("create", p))
for p in files_to_modify:
    targets.append(("modify", p))

if not targets:
    raise Exception("❌ Plan contains no files to create/modify.")

# ---- Tests first ----
def is_test(p):
    return p.replace("\\", "/").startswith("src/test/java/")

targets = sorted(targets, key=lambda t: (0 if is_test(t[1]) else 1, t[1]))
targets = targets[:5]  # guardrail

print("Targets (tests first):")
for action, p in targets:
    print("-", action, p)

# ---- Prompt template ----
file_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Spring Boot engineer. Output ONLY raw Java file content. No markdown. No explanations."),
    ("human",
     """Generate the FULL content for EXACTLY ONE file.

PATH: {path}
ACTION: {action}

Rules:
- Output ONLY the file content.
- No markdown fences.
- No commentary.
- Keep changes minimal and consistent with existing project style.
- For tests: prefer WebMvcTest(MockMvc).
- For controllers: avoid try/catch.

Repo context:
{repo_ctx}

Existing file content:
{existing}

Plan JSON:
{plan_json}
""")
])

new_generated = {}

for action, path in targets:
    abs_path = PROJECT_ROOT / path
    existing = ""
    if abs_path.exists():
        existing = abs_path.read_text(encoding="utf-8", errors="replace")[:6000]

    msg = file_prompt.format_messages(
        path=path,
        action=action,
        repo_ctx=json.dumps(repo_ctx, ensure_ascii=False, indent=2),
        existing=existing,
        plan_json=json.dumps(plan, ensure_ascii=False, indent=2),
    )

    raw = llm.invoke(msg).content
    content = strip_fences(raw)

    java_safety(path, content)

    new_generated[path] = content
    print(f"✅ Prepared {path} (chars={len(content)})")

generated_files.update(new_generated)

print("\nDone. Files stored in memory (generated_files).")
print("Next: run Cell 6.")

### Cell 6 - SAFE APPLY PATCH + MVN TEST + ROLLBACK

In [ ]:
# ==========================================
# STEP 6 (v3) — APPLY + MVN TEST + SAVE ARTIFACTS + ROLLBACK
#   - Safety checks before writing
#   - Backup overwritten files
#   - Write generated files
#   - Run mvn test
#   - If failure: save artifacts (generated files + logs + meta) then rollback
# ==========================================

import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

if "generated_files" not in globals() or len(generated_files) == 0:
    raise Exception("❌ No generated_files. Run Cell 5 first.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR = PROJECT_ROOT / ".ai_backups" / RUN_ID
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PROJECT_ROOT / ".ai_artifacts" / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

overwritten, created = [], []

# -------------------------------
# SAFETY CHECKS (before writing)
# -------------------------------
for rel_path, content in generated_files.items():
    # 1) Refuse if markdown/code fences leaked into code
    if "`" in content[:300]:
        raise Exception(
            f"❌ Refusing to write {rel_path}: contains backticks/code fences near the top. "
            "Fix Cell 5 parsing/strip fences."
        )

    # 2) Refuse if it doesn't look like a Java file header (for .java files)
    if rel_path.endswith(".java"):
        head = content.lstrip()[:200]
        if not (head.startswith("package ") or head.startswith("import ") or head.startswith("public ") or head.startswith("@")):
            raise Exception(
                f"❌ Refusing to write {rel_path}: file header looks suspicious (no package/import/public/@). "
                "Model likely added explanation text."
            )

# -------------------------------
# APPLY PATCH (with backups)
# -------------------------------
for rel_path, content in generated_files.items():
    target = PROJECT_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        backup_path = BACKUP_DIR / rel_path
        backup_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, backup_path)
        overwritten.append(rel_path)
    else:
        created.append(rel_path)

    target.write_text(content, encoding="utf-8")

print(f"Backup directory: {BACKUP_DIR}")
print("Artifacts directory: ", ARTIFACT_DIR)
print("Overwritten:", overwritten if overwritten else "None")
print("Created:", created if created else "None")

# -------------------------------
# RUN TESTS
# -------------------------------
result = subprocess.run(
    ["mvn", "test"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

last_mvn_stdout = result.stdout
last_mvn_stderr = result.stderr
last_mvn_code = result.returncode

print("\nExit code:", last_mvn_code)
print("\n--- STDOUT (last 60 lines) ---")
print("\n".join(last_mvn_stdout.splitlines()[-60:]))
print("\n--- STDERR (last 60 lines) ---")
print("\n".join(last_mvn_stderr.splitlines()[-60:]))

# -------------------------------
# SAVE ARTIFACTS ALWAYS (useful even on success)
# -------------------------------
meta = {
    "run_id": RUN_ID,
    "overwritten": overwritten,
    "created": created,
    "mvn_exit_code": last_mvn_code,
}
(ARTIFACT_DIR / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "mvn_stdout.txt").write_text(last_mvn_stdout, encoding="utf-8")
(ARTIFACT_DIR / "mvn_stderr.txt").write_text(last_mvn_stderr, encoding="utf-8")

gen_dump_dir = ARTIFACT_DIR / "generated_files"
gen_dump_dir.mkdir(parents=True, exist_ok=True)

for rel_path, content in generated_files.items():
    out_path = gen_dump_dir / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(content, encoding="utf-8")

print(f"\n🧾 Saved attempt artifacts to: {ARTIFACT_DIR}")

# -------------------------------
# ROLLBACK ON FAILURE
# -------------------------------
if last_mvn_code != 0:
    print("\n❌ mvn test failed. Rolling back...")

    for rel_path in overwritten:
        backup_file = BACKUP_DIR / rel_path
        target = PROJECT_ROOT / rel_path
        if backup_file.exists():
            shutil.copy2(backup_file, target)

    for rel_path in created:
        target = PROJECT_ROOT / rel_path
        if target.exists():
            target.unlink()

    print("✅ Rollback done. Repo restored. Artifacts preserved in .ai_artifacts.")
else:
    print("\n✅ mvn test PASSED. Patch kept.")
    print("Artifacts preserved in .ai_artifacts (useful for reporting).")

In Cell 7 (self-repair) we feed the LLM three main things:

1️⃣ The failure signal (most important)

From Cell 6:

last_mvn_stdout

last_mvn_stderr

Reduced to failure_tail (last ~180–220 lines)

👉 This tells the model what broke (404 vs 400, missing method, wrong return type, etc.).

In [ ]:
# ==========================================
# STEP 7 (compact) — SELF-REPAIR (uses ONLY prior cells)
# Requires:
#   - PROJECT_ROOT (Cell 1)
#   - picked_files (Cell 4 context built from Cell 3)
#   - last_mvn_code/stdout/stderr (Cell 6)
# Produces:
#   - updates generated_files (in-memory) so you can rerun Cell 6
# ==========================================

import os, json, re
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---- prereqs ----
if "PROJECT_ROOT" not in globals():
    raise Exception("❌ PROJECT_ROOT missing. Run Cell 1.")
if "picked_files" not in globals() or not isinstance(picked_files, dict) or not picked_files:
    raise Exception("❌ picked_files missing/empty. Run Cell 4 (it builds picked_files from Cell 3).")
if "last_mvn_code" not in globals():
    raise Exception("❌ last_mvn_code missing. Run Cell 6 first.")
if last_mvn_code == 0:
    print("✅ Build already passing. Nothing to repair.")
    raise SystemExit
if "generated_files" not in globals():
    generated_files = {}

# ---- model ----
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")  # default to gpt-4o-mini
if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY in .env")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.1,
    max_tokens=900,
)

# ---- small helpers ----
def strip_fences(s: str) -> str:
    s = s.lstrip()
    s = re.sub(r"```[a-zA-Z0-9_-]*\s*\n", "", s)
    s = s.replace("\n```", "\n")
    return s.strip()

def safe_json_load(s: str):
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

def java_sanity(rel_path: str, content: str):
    if "```" in content[:300] or "`" in content[:300]:
        raise Exception(f"❌ Markdown leaked into {rel_path}")
    if rel_path.endswith(".java") and content.count("{") != content.count("}"):
        raise Exception(f"❌ {rel_path}: unbalanced braces (likely truncated)")

def insert_before_last_brace(existing: str, snippet: str) -> str:
    idx = existing.rfind("}")
    if idx == -1:
        raise Exception("❌ Cannot patch: no closing brace found.")
    snippet = snippet.strip()
    if snippet.count("{") != snippet.count("}"):
        raise Exception("❌ Patch snippet truncated (unbalanced braces).")
    out = existing[:idx].rstrip() + "\n\n" + snippet + "\n\n" + existing[idx:]
    if out.count("{") != out.count("}"):
        raise Exception("❌ Patch insertion broke brace balance.")
    return out

def is_big_risk(path: str, existing_len: int) -> bool:
    return path.endswith("ProgressTracker.java") or existing_len > 9000

# ---- inputs from Cell 6 ----
last_stdout = globals().get("last_mvn_stdout", "") or ""
last_stderr = globals().get("last_mvn_stderr", "") or ""
failure_tail = "\n".join((last_stdout + "\n" + last_stderr).splitlines()[-220:])

# ---- context from prior cells ONLY (truncate for budget) ----
# include generated_files too so that the LLM can see and repair newly created/modified files
all_ctx = {**picked_files, **generated_files}  # generated_files may be empty initially
base_ctx = {k: v[:1200] for k, v in all_ctx.items()}
plan_json = json.dumps(plan, ensure_ascii=False, indent=2) if "plan" in globals() else "{}"

# ==========================================
# A) Ask LLM for patchset JSON (no code)
# ==========================================
patchset_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY valid JSON. No markdown. No explanations. No code blocks."),
    ("human",
     """mvn test failed. Propose a minimal repair patchset.

Failure tail:\n{failure_tail}\n
Context excerpts:\n{base_ctx}\n
Plan (read-only):\n{plan_json}\n
Return JSON EXACTLY:
{{
  "modify":[{{"path":"...","reason":"...","mode":"full|patch"}}],
  "create":[{{"path":"...","reason":"...","mode":"full"}}]
}}

Rules:
- Max 3 files total.
- Only .java under src/main/java or src/test/java.
- mode="patch" means later output ONLY a small snippet to insert (e.g., one method).

HARD CONSTRAINTS:

- You may ONLY modify files that were modified or created in the previous attempt.
- You MUST NOT modify any pre-existing core project files.
- Especially DO NOT modify:
  - ProgressController
  - ProgressTracker
  - PostgreSQLCommunication
  - Any file under Services/ unless it was newly created in the last attempt.

- If you think another file must change, DO NOT propose it.
  Instead, adjust ONLY the files already modified/created previously.

- Prefer fixing tests or newly created controllers rather than touching core logic.
"""
    )
])

raw = llm.invoke(patchset_prompt.format_messages(
    failure_tail=failure_tail,
    base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
    plan_json=plan_json
)).content

patchset = safe_json_load(strip_fences(raw))
if patchset is None:
    raise Exception("❌ Could not parse patchset JSON. Raw:\n" + raw[:800])

targets = []
for it in (patchset.get("modify") or []):
    targets.append(("modify", it["path"], it.get("reason",""), it.get("mode","full")))
for it in (patchset.get("create") or []):
    targets.append(("create", it["path"], it.get("reason",""), "full"))
targets = targets[:3]
if not targets:
    raise Exception("❌ Patchset empty.")

print("✅ Patchset:")
for a,p,r,m in targets:
    print(f"- {a} ({m}): {p} | {r}")

# ==========================================
# B) Generate per-file (full or patch)
# ==========================================
full_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY raw file content. No markdown. No explanations."),
    ("human",
     """Generate FULL content for ONE file: {path}

Action: {action}
Reason: {reason}

Failure tail:\n{failure_tail}\n
Context excerpts:\n{base_ctx}\n
Plan (read-only):\n{plan_json}\n
Existing file (empty if new):\n{existing}\n
Important instructions for JSON responses and maps:
- When returning JSON maps in controller responses, prefer Map<String, Object> for body maps so values can be heterogeneous.
- Ensure any local map variable types match the declared ResponseEntity generic. Avoid using Map<String, String> when the method returns ResponseEntity<Map<String, Object>>.
"""
    )
])

patch_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY a Java snippet to INSERT into the existing class. No markdown. No explanations."),
    ("human",
     """Generate ONLY the MINIMAL Java snippet to fix: {path}

Action: {action}
Reason: {reason}

Rules:
- Output ONLY the snippet (no package/import/class header).
- No markdown fences, no extra text.

HARD CONSTRAINTS:

- You may ONLY modify files that were modified or created in the previous attempt.
- You MUST NOT modify any pre-existing core project files.
- Especially DO NOT modify:
  - ProgressController
  - ProgressTracker
  - PostgreSQLCommunication
  - Any file under Services/ unless it was newly created in the last attempt.

- If you think another file must change, DO NOT propose it.
  Instead, adjust ONLY the files already modified/created previously.

- Prefer fixing tests or newly created controllers rather than touching core logic.

Important:
- If the fix involves returning or building JSON response maps, use Map<String, Object> for the body map and ensure declared variable types match the controller's ResponseEntity generic.

Failure tail:\n{failure_tail}\n
Existing file (reference):\n{existing}\n"""
    )
])

new_generated = {}

for action, path, reason, mode in targets:
    abs_path = PROJECT_ROOT / path
    existing_full = abs_path.read_text(encoding="utf-8", errors="replace") if abs_path.exists() else ""
    mode_eff = "patch" if (action == "modify" and (mode == "patch" or is_big_risk(path, len(existing_full))) and abs_path.exists()) else "full"

    if mode_eff == "full":
        msg = full_prompt.format_messages(
            path=path,
            action=action,
            reason=reason,
            failure_tail=failure_tail,
            base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
            plan_json=plan_json,
            existing=existing_full[:8000]
        )
        content = strip_fences(llm.invoke(msg).content)
        java_sanity(path, content)
        new_generated[path] = content + ("\n" if not content.endswith("\n") else "")
        print(f"✅ FULL: {path}")

    else:
        # patch mode: file must exist (we already ensured that above), but double-check
        if not abs_path.exists():
            print(f"⚠️ File {path} missing; falling back to full generation instead of patch.")
            # regenerate using full logic
            msg = full_prompt.format_messages(
                path=path,
                action=action,
                reason=reason,
                failure_tail=failure_tail,
                base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
                plan_json=plan_json,
                existing=existing_full[:8000]
            )
            content = strip_fences(llm.invoke(msg).content)
            java_sanity(path, content)
            new_generated[path] = content + ("\n" if not content.endswith("\n") else "")
            print(f"✅ FULL (fallback): {path}")
        else:
            msg = patch_prompt.format_messages(
                path=path,
                action=action,
                reason=reason,
                failure_tail=failure_tail,
                existing=existing_full[:8000]
            )
            snippet = strip_fences(llm.invoke(msg).content)
            if "```" in snippet[:200] or "`" in snippet[:200]:
                raise Exception(f"❌ Markdown in patch snippet for {path}")
            patched = insert_before_last_brace(existing_full, snippet)
            java_sanity(path, patched)
            new_generated[path] = patched + ("\n" if not patched.endswith("\n") else "")
            print(f"✅ PATCH: {path}")

# Quick auto-fix for a common generic mismatch: if the generated file
# declares a ResponseEntity<Map<String, Object>> but uses Map<String, String>
# for the response body, replace the local map type to Map<String, Object>.
for p, content in list(new_generated.items()):
    try:
        if 'ResponseEntity<Map<String, Object>>' in content and 'Map<String, String>' in content:
            fixed = re.sub(r'Map\s*<\s*String\s*,\s*String\s*>', 'Map<String, Object>', content)
            new_generated[p] = fixed
            print(f"🔧 Auto-fixed generics in: {p}")
    except Exception:
        pass

generated_files.update(new_generated)

print("\nDone. Added/updated in generated_files:")
for k in new_generated.keys():
    print("-", k)

print("\nNow rerun Cell 6.")

# init once
if "repair_iterations" not in globals():
    repair_iterations = 0

# inside each repair attempt
repair_iterations += 1
print(f"\n🔄 Repair iteration: {repair_iterations}")

In [ ]:
# ==========================================
# STEP 8 — EVALUATION REPORT (RULE-BASED)
#   - Deterministic, no external APIs
#   - Uses pipeline variables already available in memory
# ==========================================

import json
import re


def _safe_get(name, default=None):
    return globals().get(name, default)


def _to_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return str(x)


def _normalize_patchset(ps):
    if ps is None:
        return None
    if isinstance(ps, str):
        s = ps.strip()
        if not s:
            return None
        try:
            return json.loads(s)
        except Exception:
            return {"raw": s}
    return ps


def _extract_paths_from_obj(obj):
    paths = set()
    if obj is None:
        return paths
    if isinstance(obj, dict):
        for k, v in obj.items():
            lk = str(k).lower()
            if lk in {"file", "filepath", "path", "target", "targetfile", "filename"} and isinstance(v, str):
                paths.add(v.strip("/"))
            else:
                paths |= _extract_paths_from_obj(v)
    elif isinstance(obj, list):
        for item in obj:
            paths |= _extract_paths_from_obj(item)
    elif isinstance(obj, str):
        # Heuristic: extract path-like tokens from free text
        for m in re.findall(r"[\w\-/\.]+\.(?:java|kt|xml|yml|yaml|properties|md|json|js|ts|tsx|jsx|sql)", obj):
            paths.add(m.strip("/"))
    return paths


def _extract_feature_keywords(feature_text):
    # Heuristic: keep meaningful tokens from feature request
    stop = {
        "the", "and", "for", "with", "from", "that", "this", "then", "when", "should", "must",
        "have", "has", "into", "about", "your", "you", "are", "not", "any", "all", "new", "add",
        "update", "create", "remove", "make", "code", "feature", "request", "step"
    }
    tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_/\-]{2,}", feature_text or "")
    out = []
    for t in tokens:
        lo = t.lower()
        if lo in stop:
            continue
        if lo.startswith(("http", "www")):
            continue
        out.append(lo)

    seen = set()
    dedup = []
    for t in out:
        if t not in seen:
            dedup.append(t)
            seen.add(t)
    return dedup[:40]


# Inputs from pipeline
generated_files = _safe_get("generated_files", {}) or {}
patchset = _normalize_patchset(_safe_get("patchset", None))
plan = _safe_get("plan", None)
FEATURE_REQUEST = _safe_get("FEATURE_REQUEST", "")
last_mvn_code = _safe_get("last_mvn_code", None)
last_mvn_stdout = _safe_get("last_mvn_stdout", "") or ""
last_mvn_stderr = _safe_get("last_mvn_stderr", "") or ""
created = set(_safe_get("created", []) or [])
overwritten = set(_safe_get("overwritten", []) or [])
repository_files = _safe_get("repository_files", None)  # optional

# Repair counter from Step 7
repair_iterations = _safe_get("repair_iterations", 0)


# Derived context
mvn_text = f"{last_mvn_stdout}\n{last_mvn_stderr}".lower()
patch_paths = _extract_paths_from_obj(patchset)
plan_paths = _extract_paths_from_obj(plan)
generated_paths = set(generated_files.keys())

changed_files = set()
changed_files |= generated_paths
changed_files |= patch_paths
changed_files |= created
changed_files |= overwritten


# 1) Compiles
compile_error_patterns = [
    "compilation error",
    "compilation failure",
    "cannot find symbol",
    "package does not exist",
    "symbol:   class",
    "symbol:   method",
]
compile_success_patterns = [
    "build success",
    "compiling ",
    "maven-compiler-plugin",
    "tests run:",
]

has_compile_error = any(p in mvn_text for p in compile_error_patterns)
has_compile_success_signal = any(p in mvn_text for p in compile_success_patterns) or (last_mvn_code == 0)
compiles = "No" if has_compile_error else ("Yes" if has_compile_success_signal else "No")


# 2) Tests pass
tests_pass = "Yes" if (last_mvn_code == 0 and "there are test failures" not in mvn_text) else "No"


# 3) Meets requirements
feature_text = _to_text(FEATURE_REQUEST)
evidence_text = (
    _to_text(plan).lower() + "\n" +
    _to_text(patchset).lower() + "\n" +
    "\n".join((generated_files or {}).values()).lower()
)

keywords = _extract_feature_keywords(feature_text)
if keywords:
    matched = [k for k in keywords if k in evidence_text]
    coverage = len(matched) / max(1, len(keywords))
else:
    matched, coverage = [], 0.0  # Heuristic fallback when feature request is missing

if compiles == "No" and tests_pass == "No" and coverage < 0.30:
    meets_requirements = "No"
elif coverage >= 0.70 and tests_pass == "Yes":
    meets_requirements = "Fully"
elif coverage >= 0.30:
    meets_requirements = "Partially"
else:
    meets_requirements = "No"


# 4) Unnecessary modifications
# Heuristic: expected files inferred from plan/patchset paths
expected_files = set()
expected_files |= patch_paths
expected_files |= plan_paths

if expected_files:
    extra_files = {f for f in changed_files if f not in expected_files}
else:
    # Heuristic fallback: if no expected set is available, keep risk low by default
    extra_files = set()

extra_count = len(extra_files)
total_changed = max(1, len(changed_files))
extra_ratio = extra_count / total_changed

if extra_count == 0:
    unnecessary_modifications = "Low"
elif extra_count <= 2 or extra_ratio <= 0.20:
    unnecessary_modifications = "Medium"
else:
    unnecessary_modifications = "High"


# 5) Hallucinated APIs
hallucination_patterns = [
    "cannot find symbol",
    "symbol:   method",
    "symbol:   class",
    "package does not exist",
    "classnotfoundexception",
    "nosuchmethoderror",
]
hallucinated_apis = "Yes" if any(p in mvn_text for p in hallucination_patterns) else "No"


# 6) Repair iterations needed
# Exact metric from explicit counter maintained by Step 7.
repair_iterations_needed = int(repair_iterations) if isinstance(repair_iterations, int) and repair_iterations >= 0 else 0


# 7) Final outcome
if compiles == "Yes" and tests_pass == "Yes" and meets_requirements in {"Fully", "Partially"}:
    final_outcome = "Success"
elif compiles == "No" and tests_pass == "No" and meets_requirements == "No":
    final_outcome = "Failure"
else:
    final_outcome = "Partial success"


# Report
evaluation_report = {
    "metrics": {
        "Compiles": compiles,
        "Tests pass": tests_pass,
        "Meets requirements": meets_requirements,
        "Unnecessary modifications": unnecessary_modifications,
        "Hallucinated APIs": hallucinated_apis,
        "Repair iterations needed": repair_iterations_needed,
        "Final outcome": final_outcome,
    },
    "explanations": {
        "Compiles": "Based on Maven logs: compile errors => No, otherwise build/compile signals or exit code 0 => Yes.",
        "Tests pass": "Yes only if Maven exit code is 0 and no explicit test-failure signal appears in logs.",
        "Meets requirements": "Rule-based keyword coverage between FEATURE_REQUEST and generated evidence (plan/patch/code), combined with build/test status.",
        "Unnecessary modifications": "Heuristic comparison of changed files vs expected files inferred from patchset/plan paths.",
        "Hallucinated APIs": "Detected via missing-symbol or missing-package patterns in Maven logs.",
        "Repair iterations needed": "Exact counter value from repair loop (Step 7).",
        "Final outcome": "Success if compile+tests pass and requirements are at least partially met; Failure if build/test fail and requirements not met; otherwise Partial success.",
    },
    "details": {
        "changed_files_count": len(changed_files),
        "expected_files_count": len(expected_files),
        "extra_files_count": extra_count,
        "extra_files_sample": sorted(list(extra_files))[:10],
        "requirements_keywords_count": len(keywords),
        "requirements_keywords_matched": len(matched),
        "requirements_coverage_ratio": round(coverage, 3),
    },
    "final_interpretation": (
        f"{final_outcome}: compile={compiles}, tests={tests_pass}, requirements={meets_requirements}, "
        f"unnecessary_mods={unnecessary_modifications}, hallucinated_apis={hallucinated_apis}, "
        f"repair_iterations={repair_iterations_needed}."
    ),
}

print("=== Evaluation Summary ===")
for k, v in evaluation_report["metrics"].items():
    print(f"- {k}: {v}")
print("\nInterpretation:")
print(evaluation_report["final_interpretation"])

evaluation_report

